Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([7], device='cuda:0')


Model Layers
============

break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [6]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


nn.Flatten
==========

We initialize the
[nn.Flatten](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.flatten.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [7]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.

In [8]:
#  The linear layer is a module that applies a linear transformation on the input using its stored weights and biases.

layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.

In [9]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[-0.1632, -0.0967,  0.0700,  0.0366, -0.6132,  0.7386,  0.3175,  0.2759,
         -0.1580,  0.2093,  0.1416,  0.4439, -0.1867, -0.0605, -0.3259,  0.1052,
          0.2648, -0.1391,  0.5227, -0.1240],
        [-0.1914, -0.1965,  0.1676, -0.0241, -0.4747,  0.8109, -0.1398,  0.3192,
         -0.1803,  0.4148,  0.0362,  0.0797,  0.2535, -0.2235, -0.5379,  0.1339,
          0.1078, -0.2467,  0.3565, -0.2168],
        [-0.2314, -0.2883, -0.0920,  0.0877, -0.6665,  0.5086,  0.1528,  0.2557,
         -0.1111,  0.2160,  0.1109,  0.1912,  0.0048, -0.0949, -0.2632,  0.1171,
         -0.0868, -0.1834,  0.3252, -0.3169]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.0000, 0.0000, 0.0700, 0.0366, 0.0000, 0.7386, 0.3175, 0.2759, 0.0000,
         0.2093, 0.1416, 0.4439, 0.0000, 0.0000, 0.0000, 0.1052, 0.2648, 0.0000,
         0.5227, 0.0000],
        [0.0000, 0.0000, 0.1676, 0.0000, 0.0000, 0.8109, 0.0000, 0.3192, 0.0000,
         0.4148, 0.0362, 0.0797, 0.2535, 0.0000, 0.00

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [10]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [11]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [12]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0074, -0.0061,  0.0335,  ..., -0.0243,  0.0207, -0.0355],
        [-0.0218,  0.0067, -0.0175,  ..., -0.0304,  0.0156,  0.0318]],
       device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0034, 0.0234], device='cuda:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0109,  0.0304, -0.0308,  ..., -0.0385,  0.0238,  0.0229],
        [ 0.0317, -0.0367, -0.0179,  ...,  0.0345,  0.0335, -0.0416]],
       device='cuda:0', grad_fn=<Slic

## Summary

This notebook covers how to build a neural network with `torch.nn`:

1. **`nn.Module` basics** — every network subclasses `nn.Module`, defining layers in `__init__` and the forward pass in `forward()`; the model is moved to the available accelerator (CUDA/MPS/CPU).
2. **`NeuralNetwork` model** — flattens a 28x28 image and passes it through a `Linear → ReLU` stack (784→512→512→10) to produce class logits.
3. **Layer-by-layer breakdown** — using a sample batch of images:
   - `nn.Flatten` — collapses each 2D image into a 784-length vector, keeping the batch dimension.
   - `nn.Linear` — applies a learned linear transformation (weights + bias).
   - `nn.ReLU` — introduces non-linearity by zeroing out negative values.
   - `nn.Sequential` — chains modules together so data flows through them in order.
   - `nn.Softmax` — converts final logits into per-class probabilities that sum to 1.
4. **Model parameters** — `model.named_parameters()` exposes every layer's learnable weights/biases (their names, shapes, and values), which are what gets updated during training.
